# FREE Alternative: simple  RAG with Ollama (No API Key Needed)

This version uses:
- **Ollama** for embeddings and LLM (runs on your computer, completely free)
- **ChromaDB** for vector storage

## Setup Steps:

1. Install Ollama from: https://ollama.ai
2. Run these commands in terminal:
   ```bash
   ollama pull llama3.2
   ollama pull nomic-embed-text
   ```
3. Run this notebook!

In [1]:
# Install required packages
!pip install -U langchain langchain-ollama langchain-chroma langchain-community chromadb pypdf

You should consider upgrading via the '/Users/I531807/Documents/Edureka/AGENTIC RAG/venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
PDF_DIR = PROJECT_ROOT / "data" / "pdfs"
PERSIST_DIR = PROJECT_ROOT / "storage" / "chroma_db_ollama"

PDF_DIR.mkdir(parents=True, exist_ok=True)
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PDF directory:", PDF_DIR)
print("Persist directory:", PERSIST_DIR)

Project root: /Users/I531807/Documents/Edureka/AGENTIC RAG
PDF directory: /Users/I531807/Documents/Edureka/AGENTIC RAG/data/pdfs
Persist directory: /Users/I531807/Documents/Edureka/AGENTIC RAG/storage/chroma_db_ollama


In [3]:
# Load PDFs
from langchain_community.document_loaders import PyPDFLoader

def load_pdfs_from_folder(pdf_dir: Path):
    all_docs = []
    pdf_files = list(pdf_dir.glob("*.pdf"))

    if not pdf_files:
        print(f"No PDF files found in: {pdf_dir}")
        return all_docs

    for pdf_file in pdf_files:
        print(f"Loading PDF: {pdf_file}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        all_docs.extend(docs)

    return all_docs

raw_docs = load_pdfs_from_folder(PDF_DIR)
print("Total loaded pages/documents:", len(raw_docs))

/Users/I531807/Documents/Edureka/AGENTIC RAG/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading PDF: /Users/I531807/Documents/Edureka/AGENTIC RAG/data/pdfs/Agentic_RAG_Advanced_Guide.pdf
Loading PDF: /Users/I531807/Documents/Edureka/AGENTIC RAG/data/pdfs/RAG_Introduction_and_Fundamentals.pdf
Total loaded pages/documents: 9


In [4]:
# Split documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(raw_docs)
print("Total chunks created:", len(split_docs))

Total chunks created: 17


In [5]:
# Create embeddings with Ollama (FREE!)
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("✅ Ollama embedding model ready (100% FREE!)")

✅ Ollama embedding model ready (100% FREE!)


In [7]:
# Build ChromaDB vector store
from langchain_chroma import Chroma

COLLECTION_NAME = "agentic_rag_ollama_collection"

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=str(PERSIST_DIR),
    collection_name=COLLECTION_NAME
)

print("✅ Vector store created with FREE Ollama embeddings!")

✅ Vector store created with FREE Ollama embeddings!


In [8]:
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Retriever ready.")

Retriever ready.


In [9]:
# Create LLM with Ollama (FREE!)
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

print("✅ Ollama LLM ready (100% FREE!)")

✅ Ollama LLM ready (100% FREE!)


In [10]:
# Simple RAG function
def format_context(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def simple_rag(query: str, retriever, llm):
    docs = retriever.invoke(query)
    context = format_context(docs)

    prompt = f'''
You are a helpful assistant.
Answer the question using only the provided context.
If the answer is not in the context, say you do not have enough information.

Context:
{context}

Question:
{query}
'''

    response = llm.invoke(prompt)
    return {
        "query": query,
        "retrieved_docs": docs,
        "answer": response.content
    }

# Test it!
result = simple_rag("What are the main topics in these PDFs?", retriever, llm)
print(result["answer"])

I don't have enough information to answer this question. The provided context discusses papers related to Retrieval-Augmented Generation (RAG) and its applications, but it does not mention anything about analyzing PDFs or determining the main topics within them.
